# Tutorial 08 – Clustering

## Goals

After finishing this tutorial, you should:
* be able to cluster data using multiple algorithms
* be able to evaluate clustering quality
* know the limitations of clustering algorithms presented in this tutorial

## Clustering

Clustering is a task of identifying clusters of similar data points in the dataset based on distances in feature space and assigning them cluster label. It is very often computed on unlabeled data and so it is an example of unsupervised technique. Because of unknown labels there is no single correct answer. In this tutorial we will cover 4 different clustering techniques, external and internal evaluation metrics, and some other techniques. For more technical information and other algorithms see [scikit-learn documentation on clustering](https://scikit-learn.org/stable/modules/clustering.html#overview-of-clustering-methods).

## Hierarchical Agglomerative Clustering (HAC)

The basic idea of HAC is to build clusters from the bottom up by unifying two closest clusters into a single larger one. This process creates a hierarchy of clusters (dendrogram) and the final clustering is obtained by making a 'cut' in the dendrogram.

Let's generate some dummy data to test agglomerative clustering. We will make three fairly well divided clusters.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

sns.set()  # make plots nicer

np.random.seed(42)  # set seed for reproducibility

# generate three clusters
data = pd.concat(
    [
        pd.DataFrame(
            {
                "x": np.random.normal(mean_x, 0.03, 100),
                "y": np.random.normal(500, 100, 100),
                "label": i,
            }
        )
        for i, mean_x in enumerate((0.25, 0.5, 0.75))
    ]
)


def plot_clusters(data, clusters):
    """
    Plots clusters using scatter plot and color them acordingly.

    :param pd.DataFrame data: dataframe with datapoints havig columns "x" and "y"
    :param list of int clusters: cluster label for each of the datapoint
    """
    # find outliers not belonging to any cluster (only relevant for DBSCAN)
    outlier_indices = clusters == -1

    # plot points and color them by cluster/label
    ax = sns.scatterplot(
        data=data[~outlier_indices],
        x="x",
        y="y",
        hue=clusters[~outlier_indices],
        palette="muted",
        legend=False,
    )

    if outlier_indices.any():
        ax = sns.scatterplot(
            data=data[outlier_indices], x="x", y="y", color="black", ax=ax
        )
    return ax


# examine the data
plot_clusters(data, data.label)

Now let's train a model and cluster the data.

In [ ]:
from sklearn.cluster import AgglomerativeClustering

data_train = data.drop(columns=["label"])  # remove cluster label

ac = AgglomerativeClustering(n_clusters=3)  # try to find three clusters in the data
clusters = ac.fit_predict(data_train)  # save predicted clusters

Let's inspected clusters found by the model.

In [ ]:
plot_clusters(data, clusters)

That clustering is really bad and all three clusters are mixed together. It is because we forgot to normalize features. Distances between points in y-axis are multiple orders of magnitude larger than in x-axis.

<div class="alert alert-block alert-warning"><b>
    
### Exercise 1
</b></div>

Make a pipeline that will scale both features into range [0, 1] using [MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html) and then run [AgglomerativeClustering](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.AgglomerativeClustering.html) with 3 clusters. If you did everything right, you should see the same clusters (except colors might be shuffled) as in the first figure.

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler

# scale both axis to interval [0, 1]
pipeline = make_pipeline(MinMaxScaler(), AgglomerativeClustering(n_clusters=3))
better_clusters = pipeline.fit_predict(data_train)
plot_clusters(data, better_clusters)

This highlights the importance of preprocessing before running clustering. It is also necessary to decide if and what scaling to use. Having features not scaled is equivalent to having weighted distance metric where features with larger absolute values (in our case y values) are more influential.

Hierarchical clustering can also be visualized in the form of a tree of clusters (dendrogram). The height of the vertical lines is equivalent to 'distance' between two clusters being merged. If we make a cut at height between 2.1 and 3.6, we get three clusters represented by different colors in the dendrogram. These are exactly the clusters we see in the figure above.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

Z = linkage(MinMaxScaler().fit_transform(data_train), "ward")
fig = plt.figure(figsize=(25, 10))
dn = dendrogram(Z, distance_sort=True, color_threshold=3)

Now let's explore some real-life dataset and see what we can cluster. Let's load dataset about world flags. For each country, we have its basic information and then a lot of characterization of how the flag looks. 

In [ ]:
flags = pd.read_csv("https://www.fi.muni.cz/ib031/datasets/flags.csv")
flags.head(3)

Our goal will be to cluster similar flags together and evaluate hypothesis that flags of countries in the same region (`landmass` in this dataset) are similar in appearance and therefore should form good clusters.

As usual, we need to convert categorical columns to category dtype.

In [ ]:
object_columns = flags.select_dtypes(include="object")
flags[object_columns.columns] = object_columns.astype("category")

Now we can do the preprocessing and clustering. We will one-hot encode categorical values, standardize numeric values, and finally scale everything into range [0, 1]. We will use all features except country name and code as both are unique to each entry and landmass column will serve as true label.

In [ ]:
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline

pipeline = make_pipeline(
    make_column_transformer(
        (OneHotEncoder(), make_column_selector(dtype_include="category")),
        (StandardScaler(), ["area", "population"]),
        remainder="passthrough",
    ),
    MinMaxScaler(),
    AgglomerativeClustering(n_clusters=6),
)

flags_train = flags.drop(columns=["name", "country_code", "landmass"])
landmass = flags.landmass
flags_clusters = pipeline.fit_predict(flags_train)

Now we would like to gain insight into what clusters were actually found. We cannot plot the data in a straight forward scatter plot because there are many dimensions (i.e. columns). Luckily, we can reduce the dimensionality using Principal Component Analysis (PCA) we saw in tutorial 06. PCA will project high dimensional data into a smaller number of dimensions (2 in our case) and retain as much variation in the data as possible.

In [ ]:
from sklearn.decomposition import PCA

pca_pipeline = make_pipeline(
    pipeline[:-1],
    PCA(n_components=2),
)
flags_pca = pca_pipeline.fit_transform(flags_train)

In [ ]:
flags_pca = pd.DataFrame(flags_pca, columns=["x", "y"])
plot_clusters(flags_pca, flags_clusters)

This is still a bit abstract, let's add country flags. First download the archive [flags.zip](https://www.fi.muni.cz/ib031/tutorial06/assets/flags.zip) with country flags and extract the folder `flags` into the directory with this notebook. Then you should see scatter plot with flags.

In [ ]:
from matplotlib.offsetbox import AnnotationBbox, OffsetImage


def get_image(country_code):
    path = f"flags/{country_code.lower()}.png"
    return OffsetImage(plt.imread(path), zoom=1)


ax = plot_clusters(flags_pca, flags_clusters)
country_codes = flags.country_code.str.lower()
for x0, y0, country_code in zip(flags_pca.x, flags_pca.y, country_codes):
    ab = AnnotationBbox(get_image(country_code), (x0, y0), frameon=False)
    ax.add_artist(ab)

The projection did put similar looking flags closer to each other; mostly blue flags on the right, mostly red on the bottom, and mostly green in the top left corner. But the clusters were scatter around the plot, so let's see how much they are consistent with `landmass`.

There are **\*external\*** metrics for evaluation of clustering with known labels. We will use Adjusted Rand index, Adjusted Mutual Information, and V-measure. All three measures are measuring 'agreement' between the two labelings. You can read more on these measure and their mathematical formulation in [scikit-learn documentation](https://scikit-learn.org/stable/modules/clustering.html#clustering-performance-evaluation).

<div class="alert alert-block alert-warning"><b>
    
### Exercise 2
</b></div>

Evaluate clusters found in flags using [Adjusted Rand index](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html) (ARI), [Adjusted Mutual Information](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_mutual_info_score.html) (AMI) and [V-measure](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.v_measure_score.html). Cluster labels are stored in variable `flags_clusters` and use column `landmass` as true labels. Expected values of these metrics are:
* ARI: 0.168
* AMI: 0.202
* V-measure: 0.236

**\*Is it a good agreement?\***

In [ ]:
from sklearn.metrics import (
    adjusted_rand_score,
    adjusted_mutual_info_score,
    v_measure_score,
)

print("ARI:", adjusted_rand_score(landmass, flags_clusters))
print("AMI:", adjusted_mutual_info_score(landmass, flags_clusters))
print("V-measure:", v_measure_score(landmass, flags_clusters))

## K-means

K-means algorithm relies on iterative optimization of cluster centroids. Initial centroids are selected randomly, therefore, it often needs multiple restarts to find something closer to global optimum. On the other hand, it is quite fast and simple.

Let's load another synthetic dataset and try out k-means.

In [ ]:
aggregation = pd.read_csv("https://www.fi.muni.cz/ib031/datasets/aggregation.csv")
plot_clusters(aggregation, aggregation.label)

Plots below demonstrate how random centroid initialization affects final clustering of k-means algorithm. The default implementation in scikit-learn will restart the algorithm 10 times and select the best clustering from these 10 runs. To illustrate how much the clusters differ between runs, we run the algorithm only once with different seeds and plot the cluster found by the algorithm.

In [ ]:
from sklearn.cluster import KMeans

aggregation_train = aggregation.drop(columns="label")

# three runs with different random seed
for i in range(3):
    # set random state and only one run
    kmeans = KMeans(n_clusters=7, random_state=i, n_init=1)
    clusters = kmeans.fit_predict(aggregation_train)

    plot_clusters(aggregation, clusters)
    # plot centroids
    sns.scatterplot(
        x=kmeans.cluster_centers_[:, 0],
        y=kmeans.cluster_centers_[:, 1],
        marker="X",
        color="r",
        s=200,
    )
    plt.show()

Red crosses indicate final centroids of the clusters. Different random states (and by extension centroid initializations) result in different clusters.

Now the algorithm needs to select the best clustering of different runs. It does not, however, have any ground truth labels for comparison. This is also the case in most clustering applications where we have data but no clear definition of clusters. We would like the clustering algorithm to give us some concise information of what groups of similar data points are in the data.

To evaluate the clustering without any labels we need so called **\*internal\*** evaluation criteria. These are often based on judging cluster compactness (small within cluster distances) and separation (large between cluster distances). Examples of these measures are Silhouette Coefficient, Calinski-Harabasz index, and Davies-Bouldin index.

Since we do not have any ground truth labels we also do not know how many clusters should we be looking for. However, algorithms like K-means need to know the number of clusters in advance. A typical approach to navigate around this problem is to try different values of $k$ and then select the smallest well performing value. 

<div class="alert alert-block alert-warning"><b>
    
### Exercise 3
</b></div>

Use [KElbowVisualizer](https://www.scikit-yb.org/en/latest/api/cluster/elbow.html) from yellowbrick package to find the most suitable value of $k$ for [K-means](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html) model and `aggregation` dataset using [elbow method](https://en.wikipedia.org/wiki/Elbow_method_(clustering)). Use **\* Calinski-Harabasz\*** index as metric and try $k$ values **\*from 2 to 20\***.

**\*Which value of $k$ should we select?\***

In [ ]:
from yellowbrick.cluster import KElbowVisualizer

vizualizer = KElbowVisualizer(KMeans(), metric="calinski_harabasz", k=20)
vizualizer.fit(aggregation_train)
vizualizer.show()

Another common method of evaluating clusters is a Silhouette score. It can be again used to find an elbow, but it is more often used for the silhouette plot. Good clustering will have silhouette plot with bumps of similar thickness and rounded shape. Sharp spikes usually indicate bad clusters (small number of far apart data points). You can read more about this plot in [scikit-learn](https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_silhouette_analysis.html) and [yellowbrick](https://www.scikit-yb.org/en/latest/api/cluster/silhouette.html) documentations.

In [ ]:
from yellowbrick.cluster import SilhouetteVisualizer

model = KMeans(3, random_state=42)
visualizer = SilhouetteVisualizer(model, colors="yellowbrick")

visualizer.fit(aggregation_train)
visualizer.show()

## Density-Based Spatial Clustering of Applications with Noise (DBSCAN)

DBSCAN is one of the more sophisticated clustering techniques. Its basic idea is that clusters are areas with higher density of points separated by areas of lower densities. It has number of advantages:
* no assumption on cluster shapes (K-means expects convex clusters, i.e., "spheres"),
* no need to specify number of clusters in advance (K-means needs number of clusters, HAC can cluster based on distance in tree), and
* insensitivity to outliers (K-means and HAC can be affected).

You can read more on details of DBSCAN in [Scikit's User guide](https://scikit-learn.org/stable/modules/clustering.html#dbscan).

Let's demonstrate this advantages on another synthetic dataset.

In [ ]:
from sklearn.datasets import make_blobs

sns.set()

X, y = make_blobs(random_state=170, n_samples=600, centers=5)
np.random.seed(74)
transformation = np.random.normal(
    size=(2, 2)
)  # transformation matrix for streching the data
X = X @ transformation  # transform points (matrix multiplication)
diagonal = pd.DataFrame(X, columns=["x", "y"])
diagonal["label"] = y

plot_clusters(diagonal, diagonal.label)

Let's see how well K-Means and HAC perform on this dataset.

In [ ]:
diagonal_train = diagonal.drop(columns="label")
km = KMeans(n_clusters=5)
km.fit(diagonal_train)

plot_clusters(diagonal, km.labels_)
plt.show()

hac = AgglomerativeClustering(n_clusters=5)
hac.fit(diagonal_train)
plot_clusters(diagonal, hac.labels_)

Notice that clusters in the upper left corner are incorrectly recognized in both cases. This type of clusters is tricky for HAC and K-means as 'ends' of the same cluster are further apart than 'ends' of two different clusters.

<div class="alert alert-block alert-warning"><b>
    
### Exercise 4
</b></div>

Find clusters in `diagonal` dataset using [DBSCAN](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html) and plot them using `plot_clusters` function. The result should look like [this](https://www.fi.muni.cz/ib031/tutorial06/assets/dbscan_diagonal.png).

In [ ]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN()
dbscan.fit(diagonal_train)
plot_clusters(diagonal, dbscan.labels_)

Here DBSCAN produced noticeably better result. Few black dots are "outliers" that do not belong to any cluster according to DBSCAN. However, it is not always this easy.

<div class="alert alert-block alert-warning"><b>
    
### Exercise 5
</b></div>

Find clusters in `aggregation` dataset using [DBSCAN](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html) and plot them using `plot_clusters` function.

In [ ]:
dbscan = DBSCAN()
clusters = dbscan.fit_predict(aggregation_train)

plot_clusters(aggregation_train, clusters)

That is not quite what we want. There are clearly clusters not just bunch of outliers. Big disadvantage of DBSCAN is its sensitivity to hyper parameter setting.

<div class="alert alert-block alert-warning"><b>
    
### Exercise 6
</b></div>

Tweak DBSCAN's parameters so it identifies clusters correctly. Look into the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html) for more info about each parameters function. Try to come close to the clustering in [this picture](https://www.fi.muni.cz/ib031/tutorial06/assets/dbscan_aggregation.png).

In [ ]:
dbscan = DBSCAN(eps=1.4, min_samples=7)
clusters = dbscan.fit_predict(aggregation_train)
plot_clusters(aggregation_train, clusters)

## Gaussian Mixture

Gaussian mixture model fits a component for each cluster. The components are gaussians, i.e., [generalization of gauss curve](https://en.wikipedia.org/wiki/Gaussian_function#Two-dimensional_Gaussian_function) to higher dimensions. Each point is assigned to a component whose the most likely to generate such point. The probability of generating a point is given by the probability density function at that point multiplied by the components weight (fraction of training data assigned to it). This is of course a limiting factor (recall bias-variance trade-off), but this algorithm is a good choice for data sampled from Gaussian distributions (and many natural processed do produce normally distributed data). Also one cool feature is the ability to model intersecting clusters like those below.

In [ ]:
np.random.seed(42)

cross = pd.DataFrame()
x = np.random.normal(1, 1, 100)
y = np.random.normal(0, 1, 100)
cross = pd.concat([cross, pd.DataFrame({"x": x, "y": y, "label": 0})])

x = np.random.normal(0, 0.1, 100)
y = np.random.normal(0, 5, 100)
cross = pd.concat([cross, pd.DataFrame({"x": x, "y": y, "label": 1})])

plot_clusters(cross, cross.label)

<div class="alert alert-block alert-warning"><b>
    
### Exercise 7
</b></div>

Fit [GaussianMixture](https://scikit-learn.org/stable/modules/generated/sklearn.mixture.GaussianMixture.html) model on `cross` dataset and plot them using `plot_clusters` function. Save the model to variable called `gmm`. Observe that points on both sides of the "vertical" cluster belong to the same cluster.

In [ ]:
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=2)
clusters = gmm.fit_predict(cross.drop(columns="label"))
plot_clusters(cross, clusters)

This is because guassian for "horizontal" cluster is 'flatter' and has higher variance. So the density is still higher even for points to the left of the "vertical" cluster. We can visualize it using contours of fitted Gaussians.

In [ ]:
from scipy.stats import multivariate_normal

# create grid of coordinates
N = 200
xx, yy = np.meshgrid(
    np.linspace(cross.x.min() - 0.5, cross.x.max() + 0.5, N),
    np.linspace(cross.y.min() - 0.5, cross.y.max() + 0.5, N),
)

# Gaussian for lable 0
gaussian0 = multivariate_normal(gmm.means_[0], gmm.covariances_[0])
Z0 = gaussian0.pdf(np.c_[xx.ravel(), yy.ravel()])

# Gaussian for label 1
gaussian1 = multivariate_normal(gmm.means_[1], gmm.covariances_[1])
Z1 = gaussian1.pdf(np.c_[xx.ravel(), yy.ravel()])

# plot datapoints and contours of Gaussians
ax = plot_clusters(cross, cross.label)
ax.contour(xx, yy, Z0.reshape(xx.shape), colors=[sns.color_palette()[0]], alpha=0.75)
ax.contour(xx, yy, Z1.reshape(xx.shape), colors=[sns.color_palette()[1]], alpha=0.75)
plt.show()

## Buckshot

<div class="alert alert-block alert-danger"><b>
    
### (Optional) Exercise 8
</b></div>

Implement Buckshot algorithm using [AgglomerativeClustering](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.AgglomerativeClustering.html) and [KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html) from `scikit-learn` and compare its performance on `aggregation` dataset with plain AgglomerativeClustering and KMeans. 

Buckshot algorithm does the following steps:
1. Sample $\sqrt n$ examples, where $n$ is the total number of training examples.
2. Run HAC on the sample.
3. Compute centroids of custers found by HAC.
4. Use computed centroids and initial centroids for KMeans.
5. Use clusters found by the KMeans as the final clustering.

In [ ]:
from sklearn.base import BaseEstimator, ClusterMixin
from sklearn.metrics import silhouette_score


class BuckshotClustering(ClusterMixin, BaseEstimator):
    def __init__(self, n_clusters):
        self.n_clusters = n_clusters

    def fit(self, X, y=None):
        if isinstance(X, pd.DataFrame):
            X = X.values
        # select sqrt(n) samples
        n = X.shape[0]
        sample_indices = np.random.choice(np.arange(n), int(np.sqrt(n)), replace=False)
        sample_X = X[sample_indices]

        # fit HAC
        agg_clust = AgglomerativeClustering(n_clusters=self.n_clusters)
        agg_clust.fit(sample_X)

        # compute centroids
        data = pd.DataFrame(sample_X)
        data["label"] = agg_clust.labels_
        centroids = data.groupby("label").mean()

        # fit K means with custom centroids
        kmean_clust = KMeans(
            n_clusters=self.n_clusters, init=centroids.values, n_init=1
        )
        kmean_clust.fit(X)

        # technicality of framework
        self.labels_ = kmean_clust.labels_
        return self


ac = AgglomerativeClustering(n_clusters=7)
ac.fit(aggregation_train)

km = KMeans(n_clusters=7)
km.fit(aggregation_train)

bs = BuckshotClustering(n_clusters=7)
bs.fit(aggregation_train)

print("HAC: ", silhouette_score(aggregation_train, ac.labels_))
print("k-means: ", silhouette_score(aggregation_train, km.labels_))
print("buckshot: ", silhouette_score(aggregation_train, bs.labels_))